# NASWOT и качество независимо обученных архитектур MNIST

Каждую архитектуру оцениваем с новыми случайными весами, затем обучаем отдельно. NASWOT измеряется до первого шага оптимизатора.


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *


In [ ]:
N = 3
NUM_EPOCHS = 32
hyperparams = {
    "batch_size": 256,
    "expansion_thresholds": {"fc_output": 0.4},
    "max_new_edges_per_expansion": 2000,
    "learning_rate": 1e-1,
    "weight_decay": 0,
    "t_max": 16,
}


In [ ]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc_output = nn.Linear(input_size, 10)


    def forward(self, x):
        x = self.fc_output(x)
        return x


In [ ]:
import copy
import pandas as pd
import matplotlib.pyplot as plt
from itertools import islice
from torch.optim.lr_scheduler import CosineAnnealingLR
from ssonn.train.train import expand_model, train_one_epoch, eval_one_epoch, weights_init, apply_sparsity_mask

torch.manual_seed(SEED)
model = SimpleFCN()
architecture_model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)
architecture_model.fc_output.dropout.p = 0.3


## NASWOT на 10 фиксированных мини-батчах

Как в исходном методе, считаем `logdet` матрицы совпадения знаков активаций. Для расширенного слоя учитываем `LeakyReLU`.


In [ ]:
@torch.inference_mode()
def naswot(model: nn.Module, x: torch.Tensor):
    K = torch.zeros((len(x), len(x)), dtype=torch.float32, device=x.device)
    n_activations = 0

    def count_patterns(_, inputs):
        nonlocal n_activations
        A = (inputs[0] > 0).flatten(1).float()
        K.add_(A @ A.T + (1 - A) @ (1 - A).T)
        n_activations += A.shape[1]

    hooks = [m.register_forward_pre_hook(count_patterns)
             for m in model.modules() if isinstance(m, (nn.ReLU, nn.LeakyReLU))]
    try:
        model.eval()(x)
    finally:
        for hook in hooks:
            hook.remove()

    _, logabsdet = torch.linalg.slogdet(K.cpu().double())
    return logabsdet.item(), n_activations


## Расширение, оценка и независимое обучение


In [ ]:
eval_batches = [x.to(device) for x, _ in islice(val_loader, 10)]
edge_finder = EdgeFinder(
    metric=AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    dataloader=finder_loader,
    device=device,
    aggregation_mode="mean",
    max_to_replace=hyperparams["max_new_edges_per_expansion"],
)
criterion = nn.CrossEntropyLoss()
score_rows, result_rows = [], []

for step in range(1, N + 1):
    expanded, new_edges = expand_model(architecture_model, edge_finder,
                                       hyperparams["expansion_thresholds"])
    if not expanded:
        raise RuntimeError(f"Расширение {step} не выбрало ни одной связи")

    architecture_model.apply(weights_init)
    candidate = copy.deepcopy(architecture_model)
    torch.manual_seed(SEED + step)
    candidate.apply(weights_init)
    scores = []
    for batch, x in enumerate(eval_batches):
        score, n_activations = naswot(candidate, x)
        scores.append(score)
        score_rows.append(dict(expansions=step, batch=batch, score=score,
                               n_activations=n_activations))

    apply_sparsity_mask(candidate)
    g.manual_seed(SEED)  # одинаковый порядок обучающих примеров для всех архитектур
    optimizer = optim.SGD(candidate.parameters(), lr=hyperparams["learning_rate"],
                          momentum=0.9, weight_decay=hyperparams["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=hyperparams["t_max"])
    best_val_accuracy, best_epoch = -1.0, 0
    for epoch in range(1, NUM_EPOCHS + 1):
        train_one_epoch(candidate, train_loader, optimizer, criterion, device)
        _, val_accuracy = eval_one_epoch(candidate, criterion, val_loader,
                                         "classification", device)
        scheduler.step()
        if val_accuracy > best_val_accuracy:
            best_val_accuracy, best_epoch = val_accuracy, epoch
        if epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(f"{step} расширений | эпоха {epoch}/{NUM_EPOCHS} | val_accuracy={val_accuracy:.4f}")

    result_rows.append(dict(expansions=step, mean_naswot=np.mean(scores),
                            best_val_accuracy=best_val_accuracy,
                            best_epoch=best_epoch, new_edges=new_edges))
    del candidate, optimizer, scheduler

score_df = pd.DataFrame(score_rows)
results_df = pd.DataFrame(result_rows)
assert len(score_df) == 10 * N and len(results_df) == N
results_df


## Корреляция NASWOT и val_accuracy

Каждая точка — отдельная архитектура. Среднее NASWOT вычислено по 10 мини-батчам; в корреляции каждый вариант учитывается один раз.


In [ ]:
x = results_df["mean_naswot"]
y = results_df["best_val_accuracy"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(x, y)
axes[0].set(xlabel="Средний NASWOT logdet", ylabel="Лучшая val_accuracy",
            title=f"Пирсон r = {x.corr(y):.3f}")

x_rank, y_rank = x.rank(), y.rank()
axes[1].scatter(x_rank, y_rank)
axes[1].set(xlabel="Ранг NASWOT", ylabel="Ранг val_accuracy",
            title=f"Спирмен ρ = {x.corr(y, method='spearman'):.3f}")

for ax, xs, ys in ((axes[0], x, y), (axes[1], x_rank, y_rank)):
    for step, xi, yi in zip(results_df["expansions"], xs, ys):
        ax.annotate(str(step), (xi, yi), xytext=(5, 5), textcoords="offset points")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
